## Instruction Dataset

In [18]:
import json

data_path = 'instruction-data.json'
with open(data_path, "r") as f:
    data = json.load(f)

print(data[0]["instruction"])
print(data[0]["input"])
print(data[0]["output"])


Evaluate the following phrase by transforming it into the spelling given.
freind --> friend
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


In [19]:
def format_input(entry):
    
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text


def format_resposne(entry):

    response_text = (
        f"\n\n### Response:\n{entry['output']}"
    )

    return response_text

In [11]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):

    def __init__(self, text_data, tokenizer):
        super().__init__()

        self.tokens = []
        self.prompt_len = []
        
        for entry in text_data:
            instruction_and_input = tokenizer.encode(format_input(entry))
            response = tokenizer.encode(format_resposne(entry))
            
            self.tokens.append(instruction_and_input + response)
            self.prompt_len.append(len(instruction_and_input) + len('\n\n### Response:\n'))

    def __getitem__(self, index):
        return self.tokens[index], self.prompt_len[index]
    
    def __len__(self):
        return len(self.tokens)

In [23]:
def collate_fn_pad_mask(batch, pad_token, ignore_token):

    inputs = []
    targets = []
    max_len = max([len(tokens) for tokens, _ in batch])

    for tokens, prompt_len in batch:

        padded = tokens + [pad_token] + [pad_token] * (max_len - len(tokens))
        input_i = torch.tensor(padded[:-1])
        target_i = torch.tensor(padded[1:])
        
        instr_mask_idx = torch.arange(prompt_len - 1)
        pad_mask_idx = torch.arange(max_len - len(tokens) + 1, max_len)
        mask_idx = torch.cat((instr_mask_idx, pad_mask_idx))

        if mask_idx.numel() > 0:
            target_i[mask_idx] = ignore_token

        inputs.append(input_i)
        targets.append(target_i)

    inputs_tensor = torch.stack(inputs)
    targets_tensor = torch.stack(targets)

    return inputs_tensor, targets_tensor

In [24]:
from functools import partial
from torch.utils.data import DataLoader
import tiktoken


tokenizer = tiktoken.get_encoding("gpt2")
eot_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
ignore_token = -100
num_workers = 0
batch_size = 8
customized_collate_fn = partial(collate_fn_pad_mask, pad_token=eot_id, ignore_token=ignore_token)

In [26]:
split = int(len(data) * 0.7)

train_data= data[:split]
test_data = data[split:]

train_dataset = InstructionDataset(train_data, tokenizer)
test_dataset = InstructionDataset(test_data, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

inputs, targets = next(iter(train_loader))
print(inputs.shape)
print(targets.shape)

torch.Size([8, 70])
torch.Size([8, 70])


## Training

In [ ]:
import torch
import torch.nn as nn
from previous_chapters import GPTModel

weights_path = "../5 Pre-training/gpt2-small-124M.pth"

config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
}

model = GPTModel(config)
model.load_state_dict(torch.load(weights_path, weights_only=True))